# La Liga Web Scraping Pipeline
## Project Overview
Before we can predict who wins a football match, we need high-quality, granular data. This project implements an end-to-end web scraping pipeline to extract multi-season match results and detailed shooting statistics for La Liga from FBref.

Rather than just downloading a static dataset, we will be web scraping the data. In this project we will handle recursive page navigation (scraping historical seasons dynamically), merge seperate data sources, manage request throttling to respect host servers, and handles missing data gracefully using exception handling.

Knowing that modern websites actively block automated data collection, in order to do this we will use Selenium. We can think of Selenium as a virtual robot that sits at your computer and browses the internet exactly like a human would. Instead of just sending a raw code request to a server, it physically opens up a real web browser (like Chrome), clicks on links, and lets the page load completely.

The final output being a clean, feature-rich dataset of 1,400+ historical matches saved directly to a CSV, fully optimized for downstream machine learning tasks.

### Scraping our First Page with Requests

In [1]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import undetected_chromedriver as uc
import time

print("1. Initializing browser...")
options = uc.ChromeOptions()
driver = uc.Chrome(options=options, version_main=150)

# Give the browser 2 seconds to fully render before steering it
time.sleep(2)

print("2. Navigating to FBref...")
driver.get("https://fbref.com/en/comps/12/La-Liga-Stats")

# Wait 5 seconds for Cloudflare
time.sleep(5) 

print("3. Grabbing HTML...")
html = driver.page_source

if html:
    print("Success! The browser stayed open and grabbed the HTML.")
else:
    print("Failed to grab HTML.")

1. Initializing browser...
2. Navigating to FBref...
3. Grabbing HTML...
Success! The browser stayed open and grabbed the HTML.


### Parsing the HTML Links with BeautifulSoup
In order to parse the HTML we are going to use the BeautifulSoup Library, which is just a popular Python library used for parsing HTML and XML documents to extract, navigate, and modify web data. We will be scraping the data the following way: 

1. **League Standings Parser**:
<br>Download the La Liga standings table, parse the HTML DOM, and use CSS selectors to isolate the specific anchor (< a >) tags belonging to individual squad pages.
2. **Form & Fixture Extractor**:
<br>Navigates to each squad's unique URL and use Pandas' read_html function to convert the Scores & Fixtures table into a DataFrame.
3. **Granular Shooting Stats Scraper**:
<br>Locates the relative link to the team's detailed shooting page, download it, drop multi-level header indexes, and extracts advanced metrics (`Shots`, `Shots on Target`, `Shot Distance`, `FK`, `PK`).


In [2]:
from bs4 import BeautifulSoup

# Hand the raw HTML off to BeautifulSoup, which acts like a magnifying glass to find specific data tags
soup = BeautifulSoup(html, features="html.parser")

# Search the HTML specifically for our data table
tables = soup.select('table.stats_table')

if len(tables) == 0:
    # If the list is empty, Cloudflare's security still caught us and hid the table
    print("Error: Website Blocked Web Scraping")
else:
    # Extract the actual table object from our BeautifulSoup list
    standings_table = tables[0]
    
    # Find all the links (anchor tags, or 'a') inside that table
    links = standings_table.find_all('a')
    
    # Loop through the links and pull out just the actual URL (the 'href' attribute)
    links = [l.get("href") for l in links]
    
    # Filter the list so we only keep URLs that point to individual squad pages
    links = [l for l in links if l and '/squads/' in l]

    # Formatting team_links to be full urls to use later
    team_links = [f"https://fbref.com{l}" for l in links]

### Extract Match Stats Using Pandas and Requests
In this part of the project, we will extract the match statistics for a specific team (I'm choosing Real Madrid because they're my favorite team) from their team-specific URL. We'll grab the HTML using the requests library and use pandas to instantly parse the match history table into a structured DataFrame. So in order to do complete this phase of the porject we will need to: 

1. **Fetching the Team HTML**:
<br>First, we will define our target team URL and send a GET request to retrieve the page's HTML content.
2. **Locate the Target Table**:
<br>We will inspect the webpage to find our target table. On the team page we are specifically looking for the "Scores & Fixtures" table.
3. **Parse the Table with Pandas**:
<br>Pandas has a function called `pd.read_html()`. This function scans an HTML string for `<table>` tags and automatically converts them into a list of DataFrame objects.
4. **Extract the DataFrame**:
<br>Because `pd.read_html()` always returns a list of DataFrames (even if only one match is found), we need to extract the first element of that list to get our actual table.

In [3]:
import pandas as pd
from io import StringIO

team_url = 'https://fbref.com/en/squads/53a2f082/Real-Madrid-Stats'

print(f"Navigating to {team_url}...")
driver.get(team_url)

# Wait 5 seconds.
time.sleep(5) 

# Grab the fully loaded HTML code from the page
team_html = driver.page_source

if 'id="matchlogs_for"' not in team_html:
    print("Error: The table ID 'matchlogs_for' is nowhere in the HTML.")
    if "Cloudflare" in team_html or "Just a moment..." in team_html:
        print("Reason: Cloudflare intercepted the request.")
    else:
        print("Reason: The page loaded, but the layout changed or FBref hid the table.")
else:
    # Use the exact HTML ID to extract the table
    matches = pd.read_html(StringIO(team_html), attrs={"id": "matchlogs_for"})
    
    # Pandas returns a list of tables. We want the first one [0].
    real_madrid_df = matches[0]
    
    # Sanity check
    print(real_madrid_df.head())

Navigating to https://fbref.com/en/squads/53a2f082/Real-Madrid-Stats...
         Date           Time          Comp         Round  Day Venue Result GF  \
0  2025-08-19  21:00 (12:00)       La Liga   Matchweek 1  Tue  Home      W  1   
1  2025-08-24  21:30 (12:30)       La Liga   Matchweek 2  Sun  Away      W  3   
2  2025-08-30  21:30 (12:30)       La Liga   Matchweek 3  Sat  Home      W  2   
3  2025-09-13  16:15 (07:15)       La Liga   Matchweek 4  Sat  Away      W  2   
4  2025-09-16  21:00 (12:00)  Champions Lg  League phase  Tue  Home      W  2   

  GA       Opponent Poss Attendance            Captain Formation  \
0  0        Osasuna   71      68407  Federico Valverde     4-3-3   
1  0         Oviedo   65      29758      Dani Carvajal   4-2-3-1   
2  1       Mallorca   58      72699  Federico Valverde   4-2-3-1   
3  1  Real Sociedad   36      36958      Dani Carvajal   4-2-3-1   
4  1   fr Marseille   43      74610  Federico Valverde   4-2-3-1   

  Opp Formation             Refe

### Extracting Match Shooting Stats Using Requests and Pandas
While the basic match statistics (win/loss, goals for/against, attendance) are useful, it's not enough for what we need. To build a better machine learning model, we need deeper performance metrics like shots, shots on target, free kicks, and penalty kicks. So in this phase of the project, we will: 

1. Locate the link to the shooting page using Beautiful Soup.
2. Scrape the shooting stats HTML using Requests.
3. Parse the shooting data table into a DataFrame using Pandas.

In [4]:
soup = BeautifulSoup(team_html, features="html.parser")
links = soup.find_all('a')

# Same logic as before
if len(links) == 0:
    # If the list is empty, Cloudflare's security still caught us and hid the table
    print("Error: Website Blocked Web Scraping")
else:
    links = [l.get("href") for l in links]
    links = [l for l in links if l and 'all_comps/shooting/' in l]

# Navigating to the shooting page
driver.get(f"https://fbref.com{links[0]}")

# Waiting 5 seconds
time.sleep(5)

shooting_html = driver.page_source

# Add [0] to the end to extract the dataframe from the list
shooting = pd.read_html(StringIO(shooting_html), match="Shooting")[0]

# Sanity Check
print(shooting.head())

  For Real Madrid                                                        \
             Date           Time          Comp         Round  Day Venue   
0      2025-08-19  21:00 (12:00)       La Liga   Matchweek 1  Tue  Home   
1      2025-08-24  21:30 (12:30)       La Liga   Matchweek 2  Sun  Away   
2      2025-08-30  21:30 (12:30)       La Liga   Matchweek 3  Sat  Home   
3      2025-09-13  16:15 (07:15)       La Liga   Matchweek 4  Sat  Away   
4      2025-09-16  21:00 (12:00)  Champions Lg  League phase  Tue  Home   

                              Standard                                     \
  Result GF GA       Opponent      Gls  Sh SoT  SoT%  G/Sh G/SoT PK PKatt   
0      W  1  0        Osasuna        1  18   5  27.8  0.00  0.00  1     1   
1      W  3  0         Oviedo        3  26  10  38.5  0.12  0.30  0     0   
2      W  2  1       Mallorca        2  17   7  41.2  0.12  0.29  0     0   
3      W  2  1  Real Sociedad        2  16   6  37.5  0.13  0.33  0     0   
4      W  2 

### Cleaning and Merging Our Scraped Data with Pandas
Now we have two separate data frames:

1. `real_madrid_df`: The basic match details (date, opponent, result, goals).
2. `shooting`: The detailed shooting stats (shots, shots on target, penalties).

Our goal is to clean up the shooting data and merge it with our basic match data so everything is in one clean, easy-to-use table.

In [5]:
shooting.columns = shooting.columns.droplevel()
# merging df's together
real_madrid_data = real_madrid_df.merge(shooting[["Date", "Sh", "SoT", "PK", "PKatt"]], on="Date")

# Sanity Check
print(real_madrid_data.head())

         Date           Time          Comp         Round  Day Venue Result GF  \
0  2025-08-19  21:00 (12:00)       La Liga   Matchweek 1  Tue  Home      W  1   
1  2025-08-24  21:30 (12:30)       La Liga   Matchweek 2  Sun  Away      W  3   
2  2025-08-30  21:30 (12:30)       La Liga   Matchweek 3  Sat  Home      W  2   
3  2025-09-13  16:15 (07:15)       La Liga   Matchweek 4  Sat  Away      W  2   
4  2025-09-16  21:00 (12:00)  Champions Lg  League phase  Tue  Home      W  2   

  GA       Opponent  ...            Captain Formation Opp Formation  \
0  0        Osasuna  ...  Federico Valverde     4-3-3         3-4-3   
1  0         Oviedo  ...      Dani Carvajal   4-2-3-1         3-4-3   
2  1       Mallorca  ...  Federico Valverde   4-2-3-1         5-3-2   
3  1  Real Sociedad  ...      Dani Carvajal   4-2-3-1       4-2-3-1   
4  1   fr Marseille  ...  Federico Valverde   4-2-3-1       4-2-3-1   

              Referee  Match Report Notes  Sh SoT PK PKatt  
0      Adrián Cordero  Ma

### Scraping Multiple Teams and Seasons with a Loop
Now that we know how to scrape and merge match data for a single team, we need to scale this up. To build a good machine learning model, we need data for every team in the La Liga across multiple seasons.

To do this, we will write a `for` loop. This loop will:

1. Start at the current season's standings page.
2. Grab the links for all 20 teams.
3. Loop through each team to scrape their basic match statistics and their shooting statistics.
4. Merge them together, clean the data, and tag them with the correct Team and Season identifiers.
5. Grab the link for the Previous Season and repeat the entire process.

In [6]:
years = list(range(2026, 2024, -1))

all_matches = []
standings_url = "https://fbref.com/en/comps/12/La-Liga-Stats"

for year in years:    
    # Using same logic as before
    driver.get(standings_url)
    time.sleep(5) 
    html = driver.page_source
    
    soup = BeautifulSoup(html, features="html.parser")
    standings_table = soup.select('table.stats_table')[0]
    
    links = [l.get("href") for l in standings_table.find_all('a')]
    links = [l for l in links if l and '/squads/' in l]
    team_urls = [f"https://fbref.com{l}" for l in links]
    
    previous_season = soup.select("a.prev")[0].get("href")
    standings_url = f"https://fbref.com{previous_season}"
    
    for team_url in team_urls:
        team_name = team_url.split("/")[-1].replace("-Stats", "").replace("-", " ")
        print(f"  Scraping {team_name}...")
        
        # Getting Match Stats
        driver.get(team_url)
        time.sleep(5)
        team_html = driver.page_source
        
        # Using StringIO and our targeted table ID
        matches = pd.read_html(StringIO(team_html), attrs={"id": "matchlogs_for"})[0]
        
        soup = BeautifulSoup(team_html, features="html.parser")
        links = [l.get("href") for l in soup.find_all('a')]
        links = [l for l in links if l and 'all_comps/shooting/' in l]
        
        # Getting Shooting Stats
        driver.get(f"https://fbref.com{links[0]}")
        time.sleep(5)
        shooting_html = driver.page_source
        
        shooting = pd.read_html(StringIO(shooting_html), match="Shooting")[0]
        shooting.columns = shooting.columns.droplevel()
        
        try:
            team_data = matches.merge(shooting[["Date", "Sh", "SoT", "PK", "PKatt"]], on="Date")
        except ValueError:
            continue
            
        # Filter for La Liga matches only
        team_data = team_data[team_data["Comp"] == "La Liga"]
        
        team_data["Season"] = year
        team_data["Team"] = team_name
        all_matches.append(team_data)

# Combine into one massive DataFrame and standardize column names
match_df = pd.concat(all_matches)
match_df.columns = [c.lower() for c in match_df.columns]

print(match_df.head())

  Scraping Barcelona...
  Scraping Real Madrid...
  Scraping Villarreal...
  Scraping Atletico Madrid...
  Scraping Real Betis...
  Scraping Celta Vigo...
  Scraping Getafe...
  Scraping Rayo Vallecano...
  Scraping Valencia...
  Scraping Real Sociedad...
  Scraping Espanyol...
  Scraping Athletic Club...
  Scraping Sevilla...
  Scraping Alaves...
  Scraping Elche...
  Scraping Levante...
  Scraping Osasuna...
  Scraping Mallorca...
  Scraping Girona...
  Scraping Oviedo...
  Scraping Barcelona...
  Scraping Real Madrid...
  Scraping Atletico Madrid...
  Scraping Athletic Club...
  Scraping Villarreal...
  Scraping Real Betis...
  Scraping Celta Vigo...
  Scraping Rayo Vallecano...
  Scraping Osasuna...
  Scraping Mallorca...
  Scraping Real Sociedad...
  Scraping Valencia...
  Scraping Getafe...
  Scraping Espanyol...
  Scraping Alaves...
  Scraping Girona...
  Scraping Sevilla...
  Scraping Leganes...
  Scraping Las Palmas...
  Scraping Valladolid...
         date           time     

### Exporting the Dataframe
After successfully getting all the neccesary data, we can go ahead and export it to a CSV file. 

In [7]:
# Close the browser to free up system memory
driver.quit()
print("Browser session closed.")

# Save to CSV using straight quotes
match_df.to_csv("matches.csv")
print("Success: matches.csv has been saved and is ready for the ML model!") 

Browser session closed.
Success: matches.csv has been saved and is ready for the ML model!
